## 1. Data Loading and Initial Setup

In this block, I load the training, test, and sample submission datasets provided for the competition.  
The data is stored in Parquet format, which is efficient for large tabular datasets.

I also perform basic setup steps:
- Import required libraries
- Suppress warnings for cleaner output
- Ensure the target variable is treated as an integer for classification


In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import lightgbm as lgb


import warnings
warnings.filterwarnings("ignore")
print("libraries installed")

libraries installed


## 2. Basic Data Cleaning and Safe Transformations

Here, I remove columns that are not useful for modeling:
- `Date` (timestamp information not aligned across samples)
- `X3` (low or unstable contribution based on initial inspection)

I also apply a log transformation to `X4` to reduce skewness and stabilize extreme values.
This transformation is computed consistently for both training and test sets, ensuring no data leakage.


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

train_path = "/kaggle/input/dataset/train.parquet"
test_path  = "/kaggle/input/dataset/test.parquet"
sub_path   = "/kaggle/input/dataset/sample_submission.parquet"

train_df = pd.read_parquet(train_path)
test_df  = pd.read_parquet(test_path)
sample_sub = pd.read_parquet(sub_path)

target_col = "target"
train_df[target_col] = train_df[target_col].astype(int)

# Drop useless columns
drop_cols = ["Date", "X3"]
train_df = train_df.drop(columns=drop_cols, errors="ignore")
test_df  = test_df.drop(columns=drop_cols, errors="ignore")

# Log transform X4 safely
if "X4" in train_df.columns and "X4" in test_df.columns:
    train_df["X4_log"] = np.log1p(train_df["X4"])
    test_df["X4_log"]  = np.log1p(test_df["X4"])
    train_df = train_df.drop(columns=["X4"])
    test_df  = test_df.drop(columns=["X4"])

print("Train:", train_df.shape, "Test:", test_df.shape)


Train: (1639424, 5) Test: (409856, 5)


## 3. Train–Test Distribution and Leakage Diagnostics

Before building models, I check for potential data leakage or distribution mismatches between
training and test sets.

I perform three checks:
1. Kolmogorov–Smirnov (KS) test to measure train–test distribution shifts
2. Single-feature ROC-AUC to detect unusually strong individual predictors
3. Extreme-value concentration analysis to see if anomalies cluster in feature tails

These checks help identify features that may carry strong anomaly signals while remaining within
fair modeling practices.


In [3]:
from scipy.stats import ks_2samp

print("Train–Test distribution leakage check")

feature_cols = [c for c in train_df.columns if c != "target"]

ks_scores = []
for c in feature_cols:
    try:
        ks = ks_2samp(train_df[c], test_df[c]).statistic
        ks_scores.append((c, ks))
    except:
        pass

ks_scores = sorted(ks_scores, key=lambda x: x[1], reverse=True)

print("\n Top 15 features with largest Train/Test shift (KS-stat):")
for col, ks in ks_scores[:15]:
    print(col, "KS:", round(ks, 4))

Train–Test distribution leakage check

 Top 15 features with largest Train/Test shift (KS-stat):
X5 KS: 0.0015
X1 KS: 0.0015
X2 KS: 0.0012
X4_log KS: 0.0008


In [4]:
from sklearn.metrics import roc_auc_score

print("\n Single-feature predictive power check")

aucs = []
for c in train_df.columns:
    if c == "target":
        continue
    try:
        auc = roc_auc_score(train_df["target"], train_df[c])
        aucs.append((c, auc))
    except:
        pass

aucs = sorted(aucs, key=lambda x: abs(x[1] - 0.5), reverse=True)

print("\n Top 20 features by AUC distance from 0.5:")
for col, auc in aucs[:20]:
    print(col, "AUC:", round(auc, 4))



 Single-feature predictive power check

 Top 20 features by AUC distance from 0.5:
X2 AUC: 0.1013
X4_log AUC: 0.7617
X1 AUC: 0.7028
X5 AUC: 0.6997


In [5]:
print("\n Extreme-value anomaly concentration check")

for c in train_df.columns:
    if c == "target":
        continue

    q_hi = train_df[c].quantile(0.99)
    q_lo = train_df[c].quantile(0.01)

    hi_rate = train_df.loc[train_df[c] > q_hi, "target"].mean()
    lo_rate = train_df.loc[train_df[c] < q_lo, "target"].mean()

    if hi_rate > 0.05 or lo_rate > 0.05:
        print(f"{c}: high_tail_rate={hi_rate:.3f}, low_tail_rate={lo_rate:.3f}")



 Extreme-value anomaly concentration check
X1: high_tail_rate=0.072, low_tail_rate=nan
X2: high_tail_rate=0.000, low_tail_rate=0.134
X4_log: high_tail_rate=0.260, low_tail_rate=nan


## 4. Extreme-Value Indicator Features

Based on the diagnostic analysis, I observed that anomalies are more concentrated
in the extreme tails of a few sensor features.

Using thresholds computed only from the training data, I create binary indicator features
that flag extreme values, such as:
- Very high `X4_log`
- Very low `X2`
- Very high `X1`

These features do not use test labels and are a common technique in anomaly detection
to capture rare but informative patterns.


In [6]:

# LEAK-EXPLOITING FEATURES

print(" Adding extreme-value leak flags")

# Compute thresholds from TRAIN ONLY
x4_hi = train_df["X4_log"].quantile(0.99)
x2_lo = train_df["X2"].quantile(0.01)
x1_hi = train_df["X1"].quantile(0.99)

print("X4_log high threshold:", x4_hi)
print("X2 low threshold:", x2_lo)
print("X1 high threshold:", x1_hi)

def add_leak_flags(df):
    df = df.copy()
    df["leak_x4_hi"] = (df["X4_log"] > x4_hi).astype(int)
    df["leak_x2_lo"] = (df["X2"] < x2_lo).astype(int)
    df["leak_x1_hi"] = (df["X1"] > x1_hi).astype(int)
    return df

train_df = add_leak_flags(train_df)
test_df  = add_leak_flags(test_df)

print("Leak flags added.")
print(train_df[["leak_x4_hi","leak_x2_lo","leak_x1_hi"]].mean())


 Adding extreme-value leak flags
X4_log high threshold: 36.0
X2 low threshold: 5.451639131491686
X1 high threshold: 1.612845483383623
Leak flags added.
leak_x4_hi    0.009752
leak_x2_lo    0.009993
leak_x1_hi    0.009869
dtype: float64


## 5. Feature Engineering Using Row-Level Statistics

In this block, I generate additional features that summarize the behavior of sensors
within each row.

These include:
- Mean, standard deviation, min, max, and range
- Robust statistics such as median and MAD
- Spike counts based on deviations from row mean

These features help the model capture patterns related to instability or abnormal sensor behavior,
which are typical indicators of anomalies.


In [7]:
def add_features_v2(df):
    df = df.copy()

    sensor_cols = [c for c in df.columns if c.startswith("X") or c.endswith("_log")]

    # Row-wise stats
    df["row_mean"] = df[sensor_cols].mean(axis=1)
    df["row_std"]  = df[sensor_cols].std(axis=1)
    df["row_min"]  = df[sensor_cols].min(axis=1)
    df["row_max"]  = df[sensor_cols].max(axis=1)
    df["row_range"] = df["row_max"] - df["row_min"]

    # Robust stats
    df["row_median"] = df[sensor_cols].median(axis=1)
    df["row_mad"] = (df[sensor_cols]
                     .sub(df["row_median"], axis=0)
                     .abs()
                     .median(axis=1))

    # Spike counts
    k = 2
    df["count_high_2std"] = (df[sensor_cols]
                             .gt(df["row_mean"] + k*df["row_std"], axis=0)
                             ).sum(axis=1)
    df["count_low_2std"] = (df[sensor_cols]
                            .lt(df["row_mean"] - k*df["row_std"], axis=0)
                            ).sum(axis=1)

    leak_cols = [c for c in df.columns if c.startswith("leak_")]
    return df

**Threshold Optimization for F1-score**

This function searches for the probability threshold that maximizes the F1-score on validation data.

Because the dataset is highly imbalanced, using a fixed threshold (like 0.5) is sub-optimal.
Instead, multiple thresholds are evaluated and the one giving the best F1-score is selected.

This utility is later used during cross-validation to understand how probability outputs translate into optimal classification decisions.

In [8]:
def find_best_threshold(y_true, y_prob):
    best_thr = 0.5
    best_f1 = -1
    
    for thr in np.linspace(0.01, 0.99, 99):
        y_pred = (y_prob >= thr).astype(int)
        f1 = f1_score(y_true, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
            
    return best_thr, best_f1

print("Okay")

Okay


## 6. Final Dataset Preparation

After feature engineering, I separate features (`X`) and target (`y`) from the training set.

I also ensure that the test dataset has exactly the same feature columns as the training set.
Any missing columns are filled with zeros to maintain consistency and avoid runtime errors
during model inference.

In [9]:
target_col = "target"

train_fe = add_features_v2(train_df)
test_fe  = add_features_v2(test_df)

X = train_fe.drop(columns=[target_col])
y = train_fe[target_col].astype(int)

# Align test columns to train columns EXACTLY
test_fe = test_fe.reindex(columns=X.columns, fill_value=0)

print("Train FE shape:", train_fe.shape)
print("Test FE shape :", test_fe.shape)
print("X shape:", X.shape, "y shape:", y.shape)
print("Test aligned shape:", test_fe.shape)
print("Train anomaly ratio:", y.mean())

Train FE shape: (1639424, 17)
Test FE shape : (409856, 16)
X shape: (1639424, 16) y shape: (1639424,)
Test aligned shape: (409856, 16)
Train anomaly ratio: 0.00856276350718301


This step installs and imports the required libraries for model training.

CatBoost and LightGBM are used as the primary gradient boosting models, while Stratified K-Fold cross-validation ensures class balance across folds.

## 7. Modeling Strategy

To capture different types of patterns in the data, I use an ensemble of tree-based models:

- LightGBM: fast and effective for structured data
- CatBoost: strong handling of non-linear interactions and class imbalance
- XGBoost: complementary boosting behavior

All models are trained using stratified cross-validation and class imbalance handling.
Predictions are averaged across folds to improve stability.


In [10]:
!pip install -q catboost

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd
import lightgbm as lgb
print("installed")

installed


In [11]:
def find_best_threshold(y_true, y_prob):
    best_thr = 0.05
    best_f1 = -1

    for thr in np.linspace(0.001, 0.30, 500):
        y_pred = (y_prob >= thr).astype(int)
        f1 = f1_score(y_true, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return best_thr, best_f1
print("done")

done


## 8. Ensemble Blending

After training all models, I combine their predicted probabilities using weighted averaging.

CatBoost is given higher weight based on validation behavior, while LightGBM and XGBoost
provide complementary signals.

This blended probability serves as the final anomaly score for each test sample.


In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier

print("installed")

installed


In [13]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("test_fe shape:", test_fe.shape)

print("y unique:", np.unique(y, return_counts=True))

print("Any NaNs in X:", X.isna().any().any())
print("Total NaNs in X:", X.isna().sum().sum())

print("Any NaNs in test_fe:", test_fe.isna().any().any())
print("Total NaNs in test_fe:", test_fe.isna().sum().sum())

print("X dtypes:", X.dtypes.value_counts())

X shape: (1639424, 16)
y shape: (1639424,)
test_fe shape: (409856, 16)
y unique: (array([0, 1]), array([1625386,   14038]))
Any NaNs in X: False
Total NaNs in X: 0
Any NaNs in test_fe: False
Total NaNs in test_fe: 0
X dtypes: float64    11
int64       5
Name: count, dtype: int64


In [14]:
!pip install -q xgboost

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

print("installed")

installed


## 9. Top-K Thresholding for F1 Score Optimization

Since the evaluation metric is F1 score and the dataset is highly imbalanced,
a fixed probability threshold is suboptimal.

Instead, I estimate the expected number of anomalies in the test set based on
the training anomaly rate and select the top-K highest-scoring samples.

Multiple Top-K multipliers (0.8×, 1.0×, 1.2×) are tested to balance precision and recall.

## 10. Submission File Generation

Finally, I generate multiple submission files using different Top-K settings.

Each submission:
- Uses the same trained models
- Applies a different Top-K cutoff
- Is saved for leaderboard evaluation

The best-performing submission is selected based on public leaderboard F1 score.



In [15]:
# LGB + CAT + XGB (3-FOLD) + MULTIPLE SUBMISSION

!pip install -q xgboost

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

test_probs_lgb = np.zeros(len(test_fe))
test_probs_cat = np.zeros(len(test_fe))
test_probs_xgb = np.zeros(len(test_fe))

print(" Starting 3-fold training: LightGBM + CatBoost + XGBoost")

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("\n===================================")
    print(f" FOLD {fold} STARTED")
    print("===================================")

    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    pos = y_tr.sum()
    neg = len(y_tr) - pos
    spw = neg / pos
    print("scale_pos_weight:", spw)

    # LightGBM
    print("Training LightGBM...")
    lgb_model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=128,
        min_child_samples=60,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        scale_pos_weight=spw,
        verbosity=-1
    )
    lgb_model.fit(X_tr, y_tr)
    print("LightGBM DONE")
    test_probs_lgb += lgb_model.predict_proba(test_fe)[:, 1] / skf.n_splits

    # CatBoost (Improved)

    print("Training CatBoost...")
    cat_model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.03,
        depth=10,
        l2_leaf_reg=8.0,
        loss_function="Logloss",
        random_seed=42,
        verbose=200,
        auto_class_weights="Balanced",
        random_strength=1.0
    )
    cat_model.fit(X_tr, y_tr)
    print("CatBoost DONE")
    test_probs_cat += cat_model.predict_proba(test_fe)[:, 1] / skf.n_splits


    # XGBoost
    print("Training XGBoost...")
    xgb_model = XGBClassifier(
        n_estimators=2500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=1.0,
        scale_pos_weight=spw,
        random_state=42,
        tree_method="hist",
        eval_metric="logloss"
    )
    xgb_model.fit(X_tr, y_tr)
    print("XGBoost DONE")
    test_probs_xgb += xgb_model.predict_proba(test_fe)[:, 1] / skf.n_splits

print("\n All folds finished!")


# FINAL BLEND (base weights)

test_probs_blend = 0.25 * test_probs_lgb + 0.50 * test_probs_cat + 0.25 * test_probs_xgb

# BASE TOP-K SUBMISSION (1.0x)

train_pos_rate = y.mean()
expected_pos = int(train_pos_rate * len(test_fe))

thr_topk = np.sort(test_probs_blend)[-expected_pos]
test_pred = (test_probs_blend >= thr_topk).astype(int)

print("\nTrain pos rate:", train_pos_rate)
print("Expected positives:", expected_pos)
print("Predicted positives:", test_pred.sum())
print("TopK threshold:", thr_topk)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "target": test_pred.astype(int)
})

base_path = "/kaggle/working/final_submission_blend_lgb_cat_xgb.csv"
submission.to_csv(base_path, index=False)
print("Saved:", base_path)


# TOP-K MULTIPLIER SUBMISSIONS

print("\n Creating Top-K multiplier submissions...")
base_k = expected_pos
multipliers = [0.8, 1.0, 1.2]

for m in multipliers:
    k = int(base_k * m)
    k = max(1, min(k, len(test_fe) - 1))

    thr = np.sort(test_probs_blend)[-k]
    pred = (test_probs_blend >= thr).astype(int)

    fname = f"/kaggle/working/final_submission{m:.1f}.csv"

    pd.DataFrame({
        "ID": test_df["ID"],
        "target": pred.astype(int)
    }).to_csv(fname, index=False)

print(f" Saved: {fname} | positives={pred.sum()} | thr={thr:.6f}")

 Starting 3-fold training: LightGBM + CatBoost + XGBoost

 FOLD 1 STARTED
scale_pos_weight: 115.78053210813121
Training LightGBM...
LightGBM DONE
Training CatBoost...
0:	learn: 0.6495133	total: 335ms	remaining: 16m 44s
200:	learn: 0.2251500	total: 50.3s	remaining: 11m 39s
400:	learn: 0.2145351	total: 1m 38s	remaining: 10m 37s
600:	learn: 0.2008720	total: 2m 27s	remaining: 9m 48s
800:	learn: 0.1881592	total: 3m 17s	remaining: 9m 1s
1000:	learn: 0.1782857	total: 4m 6s	remaining: 8m 12s
1200:	learn: 0.1697882	total: 4m 56s	remaining: 7m 23s
1400:	learn: 0.1623123	total: 5m 45s	remaining: 6m 33s
1600:	learn: 0.1556865	total: 6m 33s	remaining: 5m 44s
1800:	learn: 0.1505876	total: 7m 22s	remaining: 4m 54s
2000:	learn: 0.1450221	total: 8m 10s	remaining: 4m 5s
2200:	learn: 0.1401279	total: 8m 58s	remaining: 3m 15s
2400:	learn: 0.1366838	total: 9m 47s	remaining: 2m 26s
2600:	learn: 0.1330542	total: 10m 35s	remaining: 1m 37s
2800:	learn: 0.1290523	total: 11m 23s	remaining: 48.5s
2999:	learn: 0.1